In [ ]:
import re
from pathlib import Path

from google.colab import drive

WHEEL_DIRECTORY = Path("/content/drive/MyDrive/data/jlens-reasoning/wheels")
REQUIREMENTS = WHEEL_DIRECTORY / "requirements-colab.txt"
COMMIT_FILE = WHEEL_DIRECTORY / "project-commit.txt"
DIRTY_FILE = WHEEL_DIRECTORY / "project-dirty.txt"

drive.mount("/content/drive")

if not COMMIT_FILE.is_file():
    raise RuntimeError(f"Missing project commit marker: {COMMIT_FILE}")
PROJECT_COMMIT = COMMIT_FILE.read_text(encoding="utf-8").strip()
if re.fullmatch(r"[0-9a-f]{40}", PROJECT_COMMIT) is None:
    raise RuntimeError("Project commit marker is invalid")
if not DIRTY_FILE.is_file():
    raise RuntimeError(f"Missing project dirty marker: {DIRTY_FILE}")
dirty_value = DIRTY_FILE.read_text(encoding="utf-8").strip()
if dirty_value not in {"true", "false"}:
    raise RuntimeError("Project dirty marker is invalid")
PROJECT_WORKING_TREE_DIRTY = dirty_value == "true"

wheels = sorted(WHEEL_DIRECTORY.glob("jlens_reasoning-*.whl"))
if not REQUIREMENTS.is_file():
    raise RuntimeError(f"Missing locked requirements: {REQUIREMENTS}")
if len(wheels) != 1:
    raise RuntimeError(
        f"Expected exactly one project wheel in {WHEEL_DIRECTORY}, found {len(wheels)}"
    )

wheel = wheels[0]
print(f"Installing locked environment from {REQUIREMENTS}")
%pip install -qq --disable-pip-version-check --requirement {REQUIREMENTS}
print(f"Installing project wheel {wheel.name}")
%pip install -qq --disable-pip-version-check --force-reinstall --no-deps {wheel}
print("Colab project installation complete")

del COMMIT_FILE, DIRTY_FILE, REQUIREMENTS, WHEEL_DIRECTORY, dirty_value, wheel, wheels

# FLenQA linear probe assets

This notebook freezes a problem-level split, extracts final-token residual states, and saves one binary linear probe per transformer layer. It does not analyze J-Lens behavior, intervene on the model, or evaluate the held-out test split.


In [ ]:
%pip install -qq --disable-pip-version-check scikit-learn

In [ ]:
import json
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import transformers
from datasets import load_from_disk
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

from experiments.jlens_readout_sanity.constants import MODEL_NAME, MODEL_PATH
from jlens_reasoning.benchmarks.flenqa.dataset import (
    build_prompt_text,
    normalize_rows,
)
from jlens_reasoning.environments.colab import initialize_colab

context = initialize_colab(enable_wandb=False, require_cuda=True)
ASSET_DIR = context.checkpoints_dir / "flenqa-probe-assets"
SPLIT_PATH = ASSET_DIR / "problem_split.json"
PROBE_PATH = ASSET_DIR / "probes.pt"
METADATA_PATH = ASSET_DIR / "metadata.json"
SPLIT_SEED = 1729
CONTEXT_SIZES = (250, 500)
C_GRID = (0.01, 0.1, 1.0, 10.0, 100.0)
ASSET_DIR.mkdir(parents=True, exist_ok=True)

## Load FLenQA and identify underlying problems

`problem_id` is the normalized `global_sample_id`. The full-dataset validator checks that each underlying problem has 40 prompt variants.

In [ ]:
dataset = load_from_disk(context.datasets_dir / "flenqa")
raw_rows = dataset["eval"] if hasattr(dataset, "keys") else dataset
rows = normalize_rows(raw_rows, full=True)

problem_records = {}
for row in rows:
    record = problem_records.setdefault(
        row.problem_id,
        {
            "problem_id": row.problem_id,
            "task": row.task,
            "label": row.label,
            "row_count": 0,
        },
    )
    assert (record["task"], record["label"]) == (row.task, row.label)
    record["row_count"] += 1

assert len(problem_records) == 300
assert {record["row_count"] for record in problem_records.values()} == {40}
problem_records = [
    problem_records[problem_id] for problem_id in sorted(problem_records)
]
problem_ids = {record["problem_id"] for record in problem_records}
problem_ids_by_task_label = Counter(
    (record["task"], record["label"]) for record in problem_records
)
problem_ids_by_task_label

## Create or load the fixed problem-level split

The split is stratified by task and gold label before any prompt variant is selected. Its assertions ensure that every variant inherits the same partition.

In [ ]:
split_fractions = {"train": 0.6, "validation": 0.2, "test": 0.2}
strata = [(record["task"], record["label"]) for record in problem_records]
if SPLIT_PATH.is_file():
    split_asset = json.loads(SPLIT_PATH.read_text(encoding="utf-8"))
    assert split_asset["seed"] == SPLIT_SEED
    assert split_asset["fractions"] == split_fractions
else:
    train_records, heldout_records = train_test_split(
        problem_records,
        test_size=0.4,
        random_state=SPLIT_SEED,
        stratify=strata,
    )
    heldout_strata = [(record["task"], record["label"]) for record in heldout_records]
    validation_records, test_records = train_test_split(
        heldout_records,
        test_size=0.5,
        random_state=SPLIT_SEED,
        stratify=heldout_strata,
    )
    split_asset = {
        "format_version": 1,
        "seed": SPLIT_SEED,
        "fractions": split_fractions,
        "stratified_by": ["task", "label"],
        "problems": {
            "train": sorted(record["problem_id"] for record in train_records),
            "validation": sorted(record["problem_id"] for record in validation_records),
            "test": sorted(record["problem_id"] for record in test_records),
        },
        "problem_metadata": {
            str(record["problem_id"]): {
                "task": record["task"],
                "label": record["label"],
            }
            for record in problem_records
        },
    }
    SPLIT_PATH.write_text(
        json.dumps(split_asset, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )

split_problem_ids = split_asset["problems"]
train_ids = set(split_problem_ids["train"])
validation_ids = set(split_problem_ids["validation"])
test_ids = set(split_problem_ids["test"])
assert train_ids.isdisjoint(validation_ids)
assert train_ids.isdisjoint(test_ids)
assert validation_ids.isdisjoint(test_ids)
assert train_ids | validation_ids | test_ids == problem_ids
assert sum(map(len, split_problem_ids.values())) == len(problem_ids)
problem_to_split = {
    problem_id: split for split, ids in split_problem_ids.items() for problem_id in ids
}
assert all(row.problem_id in problem_to_split for row in rows)
assert all(
    split_asset["problem_metadata"][str(row.problem_id)]["task"] == row.task
    and split_asset["problem_metadata"][str(row.problem_id)]["label"] == row.label
    for row in rows
)
{split: len(ids) for split, ids in split_problem_ids.items()}

## Select 250- and 500-token training examples

Only train and validation problem IDs enter the probe workflow. The test partition remains untouched.

In [ ]:
selected_rows = [
    row
    for row in rows
    if row.ctx_size_declared in CONTEXT_SIZES
    and problem_to_split[row.problem_id] in {"train", "validation"}
]
assert selected_rows
assert all(problem_to_split[row.problem_id] != "test" for row in selected_rows)
assert {row.ctx_size_declared for row in selected_rows} == set(CONTEXT_SIZES)
assert {problem_to_split[row.problem_id] for row in selected_rows} == {
    "train",
    "validation",
}
assert all(
    {
        row.ctx_size_declared
        for row in selected_rows
        if problem_to_split[row.problem_id] == split
    }
    == set(CONTEXT_SIZES)
    for split in ("train", "validation")
)
selected_counts = Counter(
    (problem_to_split[row.problem_id], row.ctx_size_declared) for row in selected_rows
)
selected_counts

## Extract final-token residual states

For each prompt, the probed position is the last input token, immediately before answer generation. Hidden states are copied to CPU float32 before fitting.

In [ ]:
causal_lm = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, dtype=torch.bfloat16, local_files_only=True
).to(context.device)
tokenizer = transformers.AutoTokenizer.from_pretrained(
    MODEL_PATH, local_files_only=True
)
causal_lm.eval()
num_layers = int(causal_lm.config.num_hidden_layers)

layer_features = [[] for _ in range(num_layers)]
example_labels = []
example_splits = []
example_metadata = []
for row in tqdm(selected_rows, desc="Extracting final-token states", unit="prompt"):
    prompt = build_prompt_text(
        task=row.task, question=row.question, mixin=row.mixin, rule=row.rule
    )
    encoded = tokenizer(prompt, return_tensors="pt", truncation=False)
    input_ids = encoded["input_ids"].to(context.device)
    attention_mask = encoded["attention_mask"].to(context.device)
    assert input_ids.ndim == 2 and input_ids.shape[0] == 1
    assert input_ids.shape[1] > 0 and input_ids.shape[1] <= 4096
    with torch.inference_mode():
        outputs = causal_lm(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            use_cache=False,
        )
    hidden_states = outputs.hidden_states
    assert hidden_states is not None and len(hidden_states) == num_layers + 1
    for layer_index in range(num_layers):
        state = (
            hidden_states[layer_index + 1][0, -1, :]
            .detach()
            .to("cpu", dtype=torch.float32)
        )
        layer_features[layer_index].append(state)
    example_labels.append(int(row.label))
    example_splits.append(problem_to_split[row.problem_id])
    example_metadata.append(
        {
            "problem_id": row.problem_id,
            "ctx_size": row.ctx_size_declared,
            "prompt_id": row.source_row_id,
        }
    )

layer_features = [torch.stack(features) for features in layer_features]
labels = np.asarray(example_labels, dtype=np.int64)
example_splits = np.asarray(example_splits)
train_mask = example_splits == "train"
validation_mask = example_splits == "validation"
assert not np.any(example_splits == "test")
assert train_mask.any() and validation_mask.any()
assert all(
    features.ndim == 2 and features.shape[0] == len(selected_rows)
    for features in layer_features
)
hidden_dim = int(layer_features[0].shape[1])
assert all(features.shape[1] == hidden_dim for features in layer_features)
{"examples": len(selected_rows), "layers": num_layers, "hidden_dim": hidden_dim}

## Train one probe per layer

Features are centered by the train mean, without per-dimension scaling. Validation log loss selects the L2 regularization strength. Positive scores mean `True`; negative scores mean `False`.

In [ ]:
probe_layers = {}
for layer_index, features in enumerate(layer_features):
    train_features = features[train_mask].numpy()
    validation_features = features[validation_mask].numpy()
    train_labels = labels[train_mask]
    validation_labels = labels[validation_mask]
    training_mean = train_features.mean(axis=0, dtype=np.float64).astype(np.float32)
    centered_train = train_features - training_mean
    centered_validation = validation_features - training_mean
    candidates = []
    for C in C_GRID:
        candidate = LogisticRegression(
            penalty="l2", solver="lbfgs", C=C, max_iter=2000, random_state=SPLIT_SEED
        )
        candidate.fit(centered_train, train_labels)
        validation_probability = candidate.predict_proba(centered_validation)[:, 1]
        candidates.append(
            (log_loss(validation_labels, validation_probability), C, candidate)
        )
    _, selected_C, probe = min(candidates, key=lambda item: (item[0], item[1]))
    train_probability = probe.predict_proba(centered_train)[:, 1]
    validation_probability = probe.predict_proba(centered_validation)[:, 1]
    weight = torch.from_numpy(probe.coef_[0].astype(np.float32, copy=True))
    bias = torch.tensor(float(probe.intercept_[0]), dtype=torch.float32)
    weight_norm = torch.linalg.vector_norm(weight)
    assert torch.isfinite(weight_norm) and weight_norm > 0
    probe_layers[layer_index] = {
        "weight": weight,
        "bias": bias,
        "unit_weight": weight / weight_norm,
        "training_mean": torch.from_numpy(training_mean),
        "C": float(selected_C),
        "train": {
            "log_loss": float(log_loss(train_labels, train_probability)),
            "accuracy": float(
                accuracy_score(train_labels, probe.predict(centered_train))
            ),
        },
        "validation": {
            "log_loss": float(log_loss(validation_labels, validation_probability)),
            "accuracy": float(
                accuracy_score(validation_labels, probe.predict(centered_validation))
            ),
        },
    }

assert set(probe_layers) == set(range(num_layers))
probe_summary = [
    {
        "layer": layer,
        "C": asset["C"],
        **{f"validation_{key}": value for key, value in asset["validation"].items()},
    }
    for layer, asset in probe_layers.items()
]
probe_summary[:3]

## Save frozen probe assets

The checkpoint contains tensors and metrics for every layer. The JSON sidecar keeps the split, extraction contract, and non-tensor metadata easy to inspect.

In [ ]:
metadata = {
    "format_version": 1,
    "project_commit": PROJECT_COMMIT,
    "model_name": MODEL_NAME,
    "model_path": MODEL_PATH,
    "split_path": str(SPLIT_PATH),
    "split": split_asset,
    "split_seed": SPLIT_SEED,
    "context_sizes": list(CONTEXT_SIZES),
    "feature_position": "final input token at index -1 before generation",
    "label_convention": {"False": 0, "True": 1, "positive_score": "True"},
    "num_layers": num_layers,
    "hidden_dim": hidden_dim,
    "example_counts": {
        "train": int(train_mask.sum()),
        "validation": int(validation_mask.sum()),
    },
    "regularization_grid": list(C_GRID),
    "probe_metrics": {
        str(layer): {
            "C": asset["C"],
            "train": asset["train"],
            "validation": asset["validation"],
        }
        for layer, asset in probe_layers.items()
    },
}
checkpoint = {
    "format_version": 1,
    "metadata": metadata,
    "layers": probe_layers,
}
torch.save(checkpoint, PROBE_PATH)
METADATA_PATH.write_text(
    json.dumps(metadata, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
assert SPLIT_PATH.is_file() and PROBE_PATH.is_file() and METADATA_PATH.is_file()
print(f"Saved split: {SPLIT_PATH}")
print(f"Saved probes: {PROBE_PATH}")
print(f"Saved metadata: {METADATA_PATH}")
print(
    {
        "train_examples": int(train_mask.sum()),
        "validation_examples": int(validation_mask.sum()),
        "layers": num_layers,
    }
)